In [0]:
from pyspark.sql import functions as F , types as T


### brands

In [0]:
brand_schema = T.StructType([
    T.StructField("brand_code",T.StringType(),False),
    T.StructField("brand_name",T.StringType(),True),
    T.StructField("Category_code",T.StringType(),True)
])

In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw/brands/*.csv"
df = spark .read.option("header",True).option("delimeter",",").schema(brand_schema).csv(raw_data_path)

# add Metadata column 
df = df.withColumn("_source_file",F.col("_metadata_file_path"))\
    .withColumn("ingested_at",F.current_timestamp())

display(df.limit(5))

brand_code,brand_name,Category_code,_source_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-09-03T16:45:00.838Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-09-03T16:45:00.838Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-09-03T16:45:00.838Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-09-03T16:45:00.838Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-09-03T16:45:00.838Z


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema",True)\
    .saveAsTable("ecommerce.bronze.brz_brands")



### Category 

In [0]:
# Define schema
category_schema = T.StructType([
    T.StructField("category_code", T.StringType(), False),
    T.StructField("category_name", T.StringType(), True)
])

# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/category/*.csv"

df_raw = spark.read.option("header", "true") \
    .option("delimiter", ",") \
    .schema(category_schema) \
    .csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.col("_metadata.file_path"))

# Write raw data to the Bronze Layer (catalog: ecommerce, schema: bronze, table: brz_category)
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("ecommerce.bronze.brz_category")

### Products

In [0]:

products_schema = T.StructType([
    T.StructField("product_id", T.StringType(), False),
    T.StructField("sku", T.StringType(), True),
    T.StructField("category_code", T.StringType(), True),
    T.StructField("brand_code", T.StringType(), True),
    T.StructField("color", T.StringType(), True),
    T.StructField("size", T.StringType(), True),
    T.StructField("material", T.StringType(), True),
    T.StructField("weight_grams", T.StringType(), True),  # datatype is string due to anomalies
    T.StructField("length_cm", T.StringType(), True),     # datatype is string due to anomalies
    T.StructField("width_cm", T.FloatType(), True),
    T.StructField("height_cm", T.FloatType(), True),
    T.StructField("rating_count", T.IntegerType(), True),
    T.StructField("file_name", T.StringType(), False),
    T.StructField("ingest_timestamp", T.TimestampType(), False)
])

# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/products/*.csv"

df = spark.read.option("header", "true") \
    .option("delimiter", ",") \
    .schema(products_schema) \
    .csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

# Write raw data to the Bronze Layer (catalog: ecommerce, schema: bronze, table: brz_products)
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("ecommerce.bronze.brz_products")


### Customers

In [0]:


customers_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("phone", T.StringType(), True),
    T.StructField("country_code", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("state", T.StringType(), True)
])

# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/customers/*.csv"

df_raw = spark.read.option("header", "true") \
    .option("delimiter", ",") \
    .schema(customers_schema) \
    .csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

# Write raw data to the Bronze Layer (catalog: ecommerce, schema: bronze, table: brz_customers)
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("ecommerce.bronze.brz_customers")


In [0]:

# Define schema for the data file
date_schema = T.StructType([
    T.StructField("date", T.StringType(), True),          # Raw date in string format
    T.StructField("year", T.IntegerType(), True),         # Year
    T.StructField("day_name", T.StringType(), True),      # Day name (can be mixed case)
    T.StructField("quarter", T.IntegerType(), True),      # Quarter
    T.StructField("week_of_year", T.IntegerType(), True), # Week of year (can be negative)
])

# Load data using the schema defined
raw_data_path = f"/Volumes/ecommerce/source_data/raw/date/*.csv"

df_raw = spark.read.option("header", "true") \
    .option("delimiter", ",") \
    .schema(date_schema) \
    .csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_ingested_at", F.current_timestamp()) \
               .withColumn("_source_file", F.col("_metadata.file_path"))

# Write raw data to the Bronze Layer (catalog: ecommerce, schema: bronze, table: brz_calendar)
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("ecommerce.bronze.brz_calendar")
